# Tool Use — Lab 1: Turning functions into tools

Give an LLM a set of Python functions as tools and watch it pick the right one.
Tools live in `src/tools.py` (each wrapped with LangChain's `@tool`); the agent
is assembled in `src/agent.py`. We try tool calling three ways: **auto** (the agent
runs tools via `create_agent`), **manual** (you handle the `tool_calls` with
`bind_tools`), and **many tools** (the agent picks).

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

from src import config
from src.tools import get_current_time, get_weather_from_ip, write_txt_file, generate_qr_code
from src.agent import build_agent
from src.rendering import print_html
from langchain_core.messages import ToolMessage

load_dotenv()

from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()


## Auto tool-calling with `create_agent`

In [ ]:
agent = build_agent([get_current_time])
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What time is it?"}]},
    config={"configurable": {"thread_id": "tool-use-1"}, "callbacks": [langfuse_handler]},
)
print(result["messages"][-1].content)
langfuse.flush()


## Manual tool-calling (under the hood with `bind_tools`)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import ToolMessage

model = init_chat_model(config.MODEL)
model_with_tools = model.bind_tools([get_current_time])

messages = [{"role": "user", "content": "What time is it?"}]
resp = model_with_tools.invoke(messages, config={"callbacks": [langfuse_handler]})

if resp.tool_calls:
    call = resp.tool_calls[0]
    result = get_current_time.invoke(call["args"])
    messages.append(resp)
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    final = model_with_tools.invoke(messages, config={"callbacks": [langfuse_handler]})
    print(final.content)

langfuse.flush()


## Many tools — the agent picks

In [ ]:
agent = build_agent([get_current_time, get_weather_from_ip, write_txt_file, generate_qr_code])

prompts = [
    "Can you get the weather for my location?",
    "Make a note at outputs/reminders.txt to call Daniel tomorrow at 7PM.",
    "Make a QR code to https://www.deeplearning.ai called dl_qr_code.",
]

def run(prompt, thread_id):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": thread_id}, "callbacks": [langfuse_handler]},
    )
    for msg in result["messages"]:
        if isinstance(msg, ToolMessage):
            print_html(msg.content, title=f"Tool: {msg.name}")
    print_html(result["messages"][-1].content, title="Final answer")

for i, prompt in enumerate(prompts):
    run(prompt, thread_id=f"many-tools-{i}")

langfuse.flush()